# Auditoría de robustez metodológica V2 — relación de aspecto, baselines y CKA

**Objetivo:** evaluar si los hallazgos del manuscrito revisado de *Journal of Imaging* se mantienen bajo alternativas al redimensionamiento directo 512×384 → 224×224.

Se evalúan tres estrategias con **las mismas imágenes originales y transformadas** y con los mismos pesos y bloques VGG-16:

- **A, `stretch_original`:** redimensionamiento directo a 224×224 (reproduce el método principal); se verifica contra las métricas originales.
- **B, `letterbox`:** 224×168, centrado sobre un lienzo 224×224 de color próximo a la media ImageNet.
- **C, `no_padding`:** 224×168 sin relleno, pasando solo por las capas convolucionales.

Para cada estrategia recalculamos **coseno y distancia euclidiana relativa**, **baselines de pares de imágenes originales del mismo subconjunto** y **CKA lineal centrado**. Usar los baselines del pipeline original para las alternativas sería incorrecto: su geometría puede cambiar.

**Ejecutar primero `MODO = "piloto"` (10 imágenes/clase).** La prueba piloto solo detecta cambios importantes; no establece robustez del estudio completo. Si los resultados son informativos, ejecutar `equilibrado822`, y finalmente `completo` cuando esté justificado.

El análisis NO modifica las imágenes, métricas originales ni repositorios. Los efectos del relleno no se confunden con los del estiramiento porque se mantienen los controles B y C por separado. Todos los resultados están sujetos a interpretación científica, no solo a p-valores.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, random, platform, json, hashlib, zipfile, math
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TVF, InterpolationMode
from torchvision.models import vgg16, VGG16_Weights
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# CONFIGURACIÓN. Ejecutar primero 'piloto'. Mover a 822 o completo según resultados.
MODO = "piloto"                    # "piloto" | "equilibrado822" | "completo"
IMAGENES_POR_CLASE = 10           # sólo para piloto; muestreo estable por clase
SEED = 2026
BATCH_SIZE = 12                  # bajar a 4 si la GPU no tiene memoria suficiente
NUM_WORKERS = 0                 # evita excepciones de multiprocessing observadas en Colab
BASE = Path('/content/drive/MyDrive/2026-0')
ORIGINAL_ROOT = BASE/'DatasetOriginalCompleto'
PNG_ROOT = BASE/'TrashNet_transformaciones_completo_png'
REF_CSV = BASE/'Resultados_VGG16_TrashNet_completo_v2_resize224'/'embeddings_metricas_VGG16_TrashNet_completo_v2.csv'
OUTPUT_ROOT = BASE/'Auditoria_Robustez_Aspecto_VGG16'/MODO
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BLOCKS = {'Block1':3,'Block2':8,'Block3':15,'Block4':22,'Block5':29}
CONDITIONS = [(t,l) for t in ['brightness','contrast','rotation','zoom'] for l in ['low','medium','high']]
METRICS = ['cosine_similarity','relative_euclidean_distance','euclidean_distance']
KEY = ['class','original_filename','transformation','level','block']
print('Dispositivo:',DEVICE,'| modo:',MODO)
print('Torch:',torch.__version__,'| torchvision:',torchvision.__version__)
assert ORIGINAL_ROOT.exists(), f'No encuentro {ORIGINAL_ROOT}'
assert PNG_ROOT.exists(), f'No encuentro {PNG_ROOT}'
assert REF_CSV.exists(), f'No encuentro {REF_CSV}'


In [ ]:
ref = pd.read_csv(REF_CSV)
assert len(ref)==151620, f'Se esperaban 151620 registros, hay {len(ref)}'
assert not ref.duplicated(KEY).any(), 'CSV original tiene claves duplicadas'
assert not ref[METRICS].isna().any().any(), 'CSV original tiene NaN'
ids = ref[['class','original_filename','in_balanced_822']].drop_duplicates().sort_values(['class','original_filename'])
assert len(ids)==2527
if MODO == 'piloto':
    rng = np.random.default_rng(SEED)
    parts=[]
    for clase,g in ids.groupby('class',sort=True):
        ix=rng.choice(len(g), size=min(IMAGENES_POR_CLASE,len(g)), replace=False)
        parts.append(g.iloc[sorted(ix)])
    selected = pd.concat(parts, ignore_index=True)
elif MODO == 'equilibrado822':
    selected = ids[ids.in_balanced_822==True].copy()
elif MODO == 'completo':
    selected = ids.copy()
else: raise ValueError('MODO inválido')
selected = selected.sort_values(['class','original_filename']).reset_index(drop=True)
print('Imágenes seleccionadas:',len(selected), '| por clase:', selected['class'].value_counts().to_dict())
selected.to_csv(OUTPUT_ROOT/'imagenes_seleccionadas.csv',index=False)
assert all((ORIGINAL_ROOT/row['class']/row.original_filename).exists() for _,row in selected.iterrows())


In [ ]:
# Configuraciones alternativas (sin modificar los PNG ni las imágenes originales).
# Modo con relleno: valor RGB aproximado a la media ImageNet; tras normalización ≈ 0.
NORM = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
MEAN_RGB = tuple(round(x*255) for x in [0.485,0.456,0.406])

def preprocess_aspect(img, strategy):
    w,h = img.size
    if strategy == 'stretch_original':
        small = TVF.resize(img,[224,224],interpolation=InterpolationMode.BILINEAR,antialias=True)
        return NORM(TVF.to_tensor(small))
    elif strategy == 'letterbox':
        scale = min(224/w,224/h)
        new_w,new_h = max(1,round(w*scale)),max(1,round(h*scale))
        small = TVF.resize(img,[new_h,new_w],interpolation=InterpolationMode.BILINEAR,antialias=True)
        canvas = Image.new('RGB',(224,224),color=MEAN_RGB)
        canvas.paste(small,((224-new_w)//2,(224-new_h)//2))
        return NORM(TVF.to_tensor(canvas))
    elif strategy == 'no_padding':
        # El modelo.features admite dimensiones espaciales variables; no ejecutar model.classifier.
        new_w = 224
        new_h = max(1,round(h*224/w))
        small = TVF.resize(img,[new_h,new_w],interpolation=InterpolationMode.BILINEAR,antialias=True)
        return NORM(TVF.to_tensor(small))
    else:
        raise ValueError(strategy)

class ImagePairs(Dataset):
    def __init__(self,condition, level, strategy):
        self.items=[]; self.strategy=strategy
        for row in selected.itertuples(index=False):
            p1=ORIGINAL_ROOT/row[0]/row[1]
            p2=PNG_ROOT/f'{condition}_{level}'/row[0]/(Path(row[1]).stem+'.png')
            if not p2.exists(): raise FileNotFoundError(str(p2))
            self.items.append((row[0],row[1],p1,p2))
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        cls,filename,p1,p2=self.items[i]
        with Image.open(p1) as im: a=preprocess_aspect(im.convert('RGB'),self.strategy)
        with Image.open(p2) as im: b=preprocess_aspect(im.convert('RGB'),self.strategy)
        return a,b,cls,filename

weights = VGG16_Weights.IMAGENET1K_V1
model = vgg16(weights=weights).to(DEVICE).eval()
model.requires_grad_(False)
activations={}; hooks=[]
for name, idx in BLOCKS.items():
    def hook(module,input,output,label=name): activations[label]=output
    hooks.append(model.features[idx].register_forward_hook(hook))
print('Modelo listo; capturamos las cinco capas de VGG-16')


In [ ]:
# Extracción independiente de A, B y C con reanudación por condición.
# También se guardan representaciones originales y CKA (no solo métricas por imagen).
strategies = ('stretch_original','letterbox','no_padding')
def cka_linear_centered(X,Y):
    X=np.asarray(X,dtype=np.float64); Y=np.asarray(Y,dtype=np.float64)
    X=X-X.mean(axis=0,keepdims=True); Y=Y-Y.mean(axis=0,keepdims=True)
    cross=X.T @ Y
    num=float(np.sum(cross*cross))
    gramx=X.T @ X; gramy=Y.T @ Y
    denom=float(np.sqrt(np.sum(gramx*gramx)*np.sum(gramy*gramy)))
    return num/denom if denom>0 else np.nan

for strategy in strategies:
    for transformation,level in CONDITIONS:
        outfile=OUTPUT_ROOT/f'{strategy}_{transformation}_{level}.csv'
        ckafile=OUTPUT_ROOT/f'cka_{strategy}_{transformation}_{level}.csv'
        originals_file=OUTPUT_ROOT/f'originales_{strategy}.npz'
        expected=len(selected)*5
        if outfile.exists() and ckafile.exists() and originals_file.exists():
            check=pd.read_csv(outfile); cka_check=pd.read_csv(ckafile)
            if len(check)==expected and len(cka_check)==5 and not check.duplicated(KEY).any() and not check[METRICS].isna().any().any():
                print('YA VERIFICADO',outfile.name,len(check)); continue
        ds=ImagePairs(transformation,level,strategy)
        loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
        rows=[]; xbank={b:[] for b in BLOCKS}; ybank={b:[] for b in BLOCKS}
        with torch.inference_mode():
            for orig,trans,classes,names in tqdm(loader,desc=f'{strategy} {transformation}_{level}'):
                n=orig.shape[0]; pair=torch.cat((orig,trans),dim=0).to(DEVICE)
                activations.clear()
                _=model.features(pair)
                for block in BLOCKS:
                    v=activations[block].mean(dim=(2,3))
                    v0,v1=v[:n],v[n:]
                    xbank[block].append(v0.cpu().numpy()); ybank[block].append(v1.cpu().numpy())
                    dist=torch.linalg.vector_norm(v0-v1,dim=1)
                    cos=F.cosine_similarity(v0,v1,dim=1,eps=1e-8)
                    rel=dist/(torch.linalg.vector_norm(v0,dim=1)+1e-8)
                    for j in range(n):
                        rows.append({'class':classes[j], 'original_filename':names[j],
                             'transformation':transformation,'level':level,'block':block,
                             'cosine_similarity':float(cos[j].item()),
                             'relative_euclidean_distance':float(rel[j].item()),
                             'euclidean_distance':float(dist[j].item()),'strategy':strategy})
        if len(rows)!=expected: raise AssertionError(f'Esperaba {expected}, encontré {len(rows)}')
        pd.DataFrame(rows).to_csv(outfile,index=False,float_format='%.10f')
        ckarows=[]; bank={}
        for b in BLOCKS:
            X=np.concatenate(xbank[b],axis=0); Y=np.concatenate(ybank[b],axis=0)
            assert X.shape[0]==len(selected) and Y.shape[0]==len(selected)
            ckarows.append({'strategy':strategy,'transformation':transformation,'level':level,
                            'block':b,'n':len(selected),'CKA':cka_linear_centered(X,Y)})
            bank[b]=X.astype('float32',copy=False)
        pd.DataFrame(ckarows).to_csv(ckafile,index=False)
        if not originals_file.exists(): np.savez_compressed(originals_file,**bank)
        print('GUARDADOS',outfile.name,len(rows),'y CKA para 5 bloques')
for handle in hooks: handle.remove()
print('Extracciones y CKA completadas')


In [ ]:
# Combinar tres ejecuciones; validar que A reproduce las métricas del estudio.
new=[]
for strategy in strategies:
    for t,l in CONDITIONS:
        f=OUTPUT_ROOT/f'{strategy}_{t}_{l}.csv'
        if not f.exists(): raise FileNotFoundError(f)
        new.append(pd.read_csv(f))
variants=pd.concat(new,ignore_index=True)
expected=len(selected)*12*5*len(strategies)
assert len(variants)==expected,(len(variants),expected)
assert not variants.duplicated(KEY+['strategy']).any()
ref_sample=ref.merge(selected[['class','original_filename']],on=['class','original_filename'],how='inner',validate='many_to_one')
assert len(ref_sample)==len(selected)*60
reference=ref_sample[KEY+METRICS].rename(columns={m:m+'_publicado' for m in METRICS})
a=variants[variants.strategy=='stretch_original'].merge(reference,on=KEY,validate='one_to_one')
checks=[]
for m in METRICS:
    d=np.abs(a[m]-a[m+'_publicado'])
    checks.append({'metrica':m,'n':len(d),'max_abs_delta':float(d.max()),'mean_abs_delta':float(d.mean()),
                   'n_mayor_1e_4':int((d>1e-4).sum())})
pd.DataFrame(checks).to_csv(OUTPUT_ROOT/'reproduccion_metodo_original.csv',index=False)
print('Comparación reproducción A vs métricas originales:')
display(pd.DataFrame(checks))
if any(x['n_mayor_1e_4']>0 for x in checks):
    print('ATENCIÓN: El método A difiere del original en más de 1e-4. Revisar antes de concluir robustez.')

variants.to_csv(OUTPUT_ROOT/'metricas_robustez_tres_estrategias.csv.gz',index=False,compression='gzip')
merged=variants.merge(reference,on=KEY,how='left',validate='many_to_one',indicator=True)
assert (merged['_merge']=='both').all()
merged=merged.drop(columns='_merge')
for m in METRICS:merged['delta_'+m]=merged[m]-merged[m+'_publicado']
merged.to_csv(OUTPUT_ROOT/'comparacion_emparejada.csv.gz',index=False,compression='gzip')


In [ ]:
# Resumen emparejado por condición, bloque y estrategia.
group_columns=['strategy','transformation','level','block']
rng=np.random.default_rng(SEED); records=[]
for key,g in merged.groupby(group_columns,sort=True):
    r=dict(zip(group_columns,key));r['n']=len(g)
    for m in ['cosine_similarity','relative_euclidean_distance']:
        d=g['delta_'+m].to_numpy(dtype=float)
        r['mean_published_'+m]=g[m+'_publicado'].mean()
        r['mean_variant_'+m]=g[m].mean()
        r['mean_delta_'+m]=float(d.mean());r['median_abs_delta_'+m]=float(np.median(np.abs(d)))
        r['max_abs_delta_'+m]=float(np.max(np.abs(d)))
        boot=np.array([np.mean(d[rng.integers(0,len(d),len(d))]) for _ in range(1000)])
        r['delta_ci95_low_'+m]=float(np.percentile(boot,2.5))
        r['delta_ci95_high_'+m]=float(np.percentile(boot,97.5))
    records.append(r)
summary=pd.DataFrame(records);summary.to_csv(OUTPUT_ROOT/'resumen_pareado_por_condicion_bloque.csv',index=False)
print('Resumen:',len(summary),'filas')
display(summary[['strategy','transformation','level','block','n','mean_delta_cosine_similarity','delta_ci95_low_cosine_similarity','delta_ci95_high_cosine_similarity']].head(15))


In [ ]:
# Estabilidad del orden por intensidad dentro de cada bloque y transformación.
# El orden esperado para similitud coseno es low >= medium >= high.
monotonic=[]
for (strategy,t,block),g in variants.groupby(['strategy','transformation','block'],sort=True):
    pv=g.pivot(index=['class','original_filename'],columns='level',values='cosine_similarity')
    if not set(['low','medium','high']).issubset(pv): continue
    pv=pv[['low','medium','high']].dropna()
    monotonic.append({'strategy':strategy,'transformation':t,'block':block,'n':len(pv),
        'fraction_monotonic_cosine':float(((pv.low>=pv.medium)&(pv.medium>=pv.high)).mean()),
        'fraction_low_medium_inversion':float((pv.low<pv.medium).mean()),
        'fraction_medium_high_inversion':float((pv.medium<pv.high).mean())})
mono=pd.DataFrame(monotonic)
mono.to_csv(OUTPUT_ROOT/'orden_intensidad_por_bloque.csv',index=False)
display(mono)

# Dependencia de los resultados dentro de cada bloque: correlación Spearman entre ambas estrategias.
correlations=[]
for (strategy,t,l,b),g in merged.groupby(group_columns,sort=True):
    rho_cos=spearmanr(g['cosine_similarity_publicado'],g['cosine_similarity']).statistic
    rho_rel=spearmanr(g['relative_euclidean_distance_publicado'],g['relative_euclidean_distance']).statistic
    correlations.append({'strategy':strategy,'transformation':t,'level':l,'block':b,'n':len(g),
      'spearman_cosine':rho_cos,'spearman_relative_euclidean':rho_rel})
corr=pd.DataFrame(correlations)
corr.to_csv(OUTPUT_ROOT/'correlaciones_mismo_bloque.csv',index=False)
print('Correlación mínima coseno:',corr.spearman_cosine.min(), '| relativa:',corr.spearman_relative_euclidean.min())


In [ ]:
# Baselines específicos para cada estrategia, calculados sobre las MISMAS imágenes seleccionadas.
# Reutilizamos índices de pares idénticos entre A/B/C; en piloto se usan todos los pares posibles.
classes=selected['class'].to_numpy(); n=len(classes)
ii,jj=np.triu_indices(n,k=1)
mask_same=classes[ii]==classes[jj]
random_pairs=np.arange(len(ii)); same_pairs=np.flatnonzero(mask_same); diff_pairs=np.flatnonzero(~mask_same)
rng_pairs=np.random.default_rng(SEED)
selected_indices={}
for typ, pool in [('random',random_pairs),('same_class',same_pairs),('different_class',diff_pairs)]:
    k=min(100000,len(pool))
    selected_indices[typ]=rng_pairs.choice(pool,size=k,replace=False)
print('Pares muestreados:',{k:len(v) for k,v in selected_indices.items()})
refs=[]
for strategy in strategies:
    feats=np.load(OUTPUT_ROOT/f'originales_{strategy}.npz')
    for b in BLOCKS:
        E=feats[b].astype(np.float64)
        norms=np.linalg.norm(E,axis=1,keepdims=True)
        E=E/np.maximum(norms,1e-12)
        for typ,idx in selected_indices.items():
            vals=np.sum(E[ii[idx]]*E[jj[idx]],axis=1)
            refs.append({'strategy':strategy,'block':b,'pair_type':typ,'pairs':len(vals),
                         'median_cosine':float(np.median(vals)),'mean_cosine':float(vals.mean())})
baselines=pd.DataFrame(refs)
baselines.to_csv(OUTPUT_ROOT/'baselines_por_estrategia.csv',index=False)
base_random=baselines[baselines.pair_type=='random'][['strategy','block','median_cosine']]
adjusted=variants.merge(base_random,on=['strategy','block'],how='left',validate='many_to_one')
adjusted['angular_adjusted']=(1-adjusted.cosine_similarity)/(1-adjusted.median_cosine)
adjusted_summary=adjusted.groupby(['strategy','transformation','level','block'],as_index=False).agg(
    n=('angular_adjusted','size'),mean_adjusted=('angular_adjusted','mean'),
    median_adjusted=('angular_adjusted','median'))
adjusted_summary.to_csv(OUTPUT_ROOT/'sensibilidad_ajustada_por_estrategia.csv',index=False)

ckalist=[]
for strategy in strategies:
    for t,l in CONDITIONS:
        ckalist.append(pd.read_csv(OUTPUT_ROOT/f'cka_{strategy}_{t}_{l}.csv'))
cka=pd.concat(ckalist,ignore_index=True)
cka.to_csv(OUTPUT_ROOT/'CKA_por_estrategia.csv',index=False)
print('Baselines por bloque (mediana de pares random):')
display(baselines[baselines.pair_type=='random'])
print('CKA por estrategia (primera parte):')
display(cka.head(12))
# Comparación cualitativa: ¿qué bloques tienen el máximo ajuste para cada transformación e intensidad?
peaks=adjusted_summary.loc[adjusted_summary.groupby(['strategy','transformation','level'])['mean_adjusted'].idxmax()].copy()
peaks.to_csv(OUTPUT_ROOT/'bloques_pico_sensibilidad_ajustada.csv',index=False)
print('Bloques con mayor sensibilidad ajustada:');display(peaks)


In [ ]:
# Exportación de auditoría. Archivo pequeño sin matrices de embeddings.
info={'modo':MODO,'images':len(selected),'rows_per_strategy':len(selected)*60,
      'strategies':list(strategies),'seed':SEED,'torch':torch.__version__,
      'torchvision':torchvision.__version__,'PIL':Image.__version__,'python':platform.python_version(),
      'model':'VGG16_Weights.IMAGENET1K_V1','layers':BLOCKS,
      'original_preprocessing':'Resize((224,224)) sin relación de aspecto',
      'letterbox':'Resize a 224×168 sobre 224×224 con padding RGB cercano a la media ImageNet',
      'no_padding':'Resize a 224×168, solo model.features',
      'baseline':'mediana coseno pares originales random por estrategia sobre el mismo conjunto de imágenes',
      'cka':'linear centered CKA por estrategia / bloque / condición',
      'limitations':'piloto pequeño no valida resultados poblacionales; controles distintos a preentrenamiento; CKA exploratorio con pocas imágenes'}
(OUTPUT_ROOT/'metadatos_robustez.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf8')
notes=[f'# Auditoría V2 de robustez ({MODO})',f'{len(selected)} imágenes, {len(selected)*60} observaciones por estrategia.',
       'Se incluyen: métricas originales recalculadas, controles B/C, referencias geométricas por estrategia y CKA.',
       'No interpretar diferencias de CKA o bloques máximos del piloto como prueba suficiente de robustez del conjunto completo.',
       'Comparar dentro del mismo conjunto de imágenes; no aplicar el baseline publicado A a estrategias B/C.',
       'Separar efectos del relleno del efecto de deformación 4:3 a 1:1.',
       'Si la reproducción A difiere sustancialmente (>1e-4), detener interpretación y revisar el código.',
       '', '## Cambios medios máximos frente a los resultados publicados (descriptivos)']
for strategy in strategies:
    s=summary[summary.strategy==strategy]
    notes.append(f'- {strategy}: max |delta media cos| = {s.mean_delta_cosine_similarity.abs().max():.7f}; rel = {s.mean_delta_relative_euclidean_distance.abs().max():.7f}')
(OUTPUT_ROOT/'LEEME_resultados.md').write_text('\n'.join(notes),encoding='utf8')
zip_path=BASE/f'Reporte_Robustez_Aspecto_VGG16_V2_{MODO}.zip'
files=['imagenes_seleccionadas.csv','metadatos_robustez.json','LEEME_resultados.md',
       'reproduccion_metodo_original.csv','resumen_pareado_por_condicion_bloque.csv',
       'orden_intensidad_por_bloque.csv','correlaciones_mismo_bloque.csv',
       'baselines_por_estrategia.csv','sensibilidad_ajustada_por_estrategia.csv',
       'CKA_por_estrategia.csv','bloques_pico_sensibilidad_ajustada.csv']
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for name in files:z.write(OUTPUT_ROOT/name,arcname=name)
from google.colab import files as colabfiles
print('REPORTE LISTO:',zip_path)
colabfiles.download(str(zip_path))
